In [1]:
import sys
import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import struct


import funciones_aux as fau

import funciones_dsa as fun_dsa
import funciones_dsa_unilateral as fun_dsa_u
import funciones_dsa_bilateral as fun_dsa_b

import funciones_plot_dsa as fun_plot

from scipy.signal import welch
from matplotlib.colors import LinearSegmentedColormap, PowerNorm
from scipy.stats import pearsonr, spearmanr


# 1. Unilateral

In [4]:
"""ruta_fa_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.f_a"
ruta_spa_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.spa"
ruta_ha_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.h_a"
ruta_ta_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.t_a"

archivo_r2a = r"C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.r2a"
"""

'ruta_fa_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.f_a"\nruta_spa_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.spa"\nruta_ha_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.h_a"\nruta_ta_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.t_a"\n\narchivo_r2a = r"C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/M-TA6m-03041035/DH03041035/L03041035.r2a"\n'

In [5]:
# este es para el bis vista que no tiene fa
ruta_spa_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_antiguo/L03041419/L03041419/L03041419.spa"
ruta_ha_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_antiguo/L03041419/L03041419/L03041419.h_a"
ruta_ta_unilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_antiguo/L03041419/L03041419/L03041419.t_a"

archivo_r2a = r"C:/Users/usuario/TFG_BIS_GIS/data/data_bis_antiguo/L03041419/L03041419/L03041419.r2a"

## 1. 1.  Archivo Espectral .f_a

In [ ]:
tiempo_fa_unilat, dsa_unilat = fau.cargar_fa_directo(ruta_fa_unilat, escalar_db=True)

print("Dimensiones de la matriz:", dsa_unilat.shape)
print("Frecuencias:", dsa_unilat.columns.min(), "a", dsa_unilat.columns.max(), "Hz")

## 1. 2. Archivo variables procesadas .spa

In [6]:
df_spa_raw = fau.procesar_spa(ruta_spa_unilat)
df_spa_unilat = fun_dsa_u.limpiar_spa_unilateral(df_spa_raw)

print("Dimensiones del archivo procesado:", df_spa_unilat.shape)

Dimensiones del archivo procesado: (160, 11)


### Fusión de los anteriores

In [ ]:
df_merge_fa = fun_dsa.alinear_spa_con_tiempo(tiempo_fa_unilat, df_spa_unilat)
sef_hor = df_merge_fa["SEF08"]
mf_hor = df_merge_fa["MEDFRQ08"]

### Cabecera

In [8]:
num_canales, fs, pendiente, offset = fau.extraer_parametros_eeg(ruta_ha_unilat)
print("Parámetros extraídos con éxito:")
print(f" - Canales: {num_canales}")
print(f" - Frecuencia (Hz): {fs}")
print(f" - Pendiente (m): {pendiente:.8f}")
print(f" - Offset (b): {offset:.4f}")

Parámetros extraídos con éxito:
 - Canales: 2
 - Frecuencia (Hz): 128
 - Pendiente (m): 0.05000000
 - Offset (b): -3234.0000


## 1. 3 Archivo ondas crudas .r2a

In [9]:
df_eeg = fun_dsa_u.leer_r2a(
    archivo_r2a,
    pendiente,
    offset,
    fs=fs
)

In [10]:
df_dsa_canal1, frecuencias_c1 = fun_dsa.crear_matriz_dsa_fft_welch_desde_eeg(
    df_eeg,
    "canal_1_uV",
    fs=128,
    ventana_seg=1,
    paso_seg=1,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5,
    modo="densidad",
    tiempo_referencia="centro"
)

df_dsa_canal2, _ = fun_dsa.crear_matriz_dsa_fft_welch_desde_eeg(
    df_eeg,
    "canal_2_uV",
    fs=128,
    ventana_seg=1,
    paso_seg=1,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5,
    modo="densidad", 
    tiempo_referencia="centro"
)

# -------------------- Depende del modo que pongamos las uds son uV^2 o uV^2/Hz (uds. potencia)--------------------

# columnas del dF generado después de la conversión por FFT -> las frecuencias
cols_freq = [c for c in df_dsa_canal1.columns if c != "tiempo_s"]

# copia del dF del canal 1 para tener esa estructura
df_dsa_media = df_dsa_canal1.copy()

# media (en float) de las potencias (en uV) de los 2 canales
# en función de las frecuencias 
pot_media = (
    df_dsa_canal1[cols_freq].to_numpy(dtype=float) +
    df_dsa_canal2[cols_freq].to_numpy(dtype=float)
) / 2

# ----------- CONVERSIÓN A DECIBELIOS (ref documentación del BIS) -----------------------------------
ref_potencia = 0.0001
df_dsa_media[cols_freq] = 10 * np.log10(
    (pot_media + 1e-12) / (ref_potencia**2)
)

In [11]:
df_spa_unilat, hora_inicio = fun_dsa.obtener_hora_inicio_desde_spa(
    df_spa_unilat
)

tiempo_eeg, dsa_eeg = fun_dsa.adaptar_dsa_reconstruida_para_plot(
    df_dsa=df_dsa_media,
    frecuencias=frecuencias_c1,
    hora_inicio=hora_inicio,
    insertar_fila_inicial_nan=False
)

df_merge_plot = fun_dsa.alinear_spa_con_tiempo(
    tiempo=tiempo_eeg,
    df_spa=df_spa_unilat
)

_, mask_total = fun_dsa.preparar_dsa_con_mask(
    tiempo=tiempo_eeg,
    dsa=dsa_eeg,
    df_merge=df_merge_plot,
    umbral_sqi=15,
    umbral_ceros=0.9
)

mask_comun = mask_total.copy()

# en las reconstrucciones se utilizan como valores mínimos y máximos 
# los percentiles más ajustados para replicar el color


Aviso: se han encontrado 3 tiempos duplicados en el .spa. Estrategia usada: last.


In [12]:
# DSA reconstruida directa con máscara común ------------------- EEG --------------------------------------------------

""" 
Copia de la dsa proveniente del eeg para incluirla en el plot
 - .loc[mask_comun.values, :]: selecciona todas las filas donde mask_comun vale True, y todas las columnas de frecuencia.
 - np.nan: como el colormap pinta los NaN en blanco, esas filas aparecerán como bandas blancas.
"""

dsa_eeg_directa_plot = dsa_eeg.copy()
dsa_eeg_directa_plot.loc[mask_comun.values, :] = np.nan

In [ ]:
# DSA original f_a con máscara común ------------------------------- FA ---------------------------------------------------

""" 
Copia de la dsa proveniente del f_a para incluirla en el plot
 - .loc[mask_comun.values, :]: selecciona todas las filas donde mask_comun vale True, y todas las columnas de frecuencia.
 - np.nan: como el colormap pinta los NaN en blanco, esas filas aparecerán como bandas blancas.
"""

dsa_fa_plot = dsa_unilat.copy()
dsa_fa_plot.loc[mask_comun.values, :] = np.nan

## 1. 4. Preparar escala de color de f_a y eeg reconstruida

In [13]:
# DSA reconstruida ------------------------------------------------ EEG -----------------------------------------------

matriz_eeg, vmin_eeg, vmax_eeg, norm_eeg, cmap_eeg = fun_dsa.preparar_escala_color_dsa(
    dsa_eeg_directa_plot,
    gamma=0.4
)

In [ ]:
# DSA original ---------------------------------------------------- FA --------------------------------------------------

# las matrices que vienen de la f_a suelen mostrar valores entre el 49 y 94
matriz_fa, vmin_fa, vmax_fa, norm_fa, cmap_fa = fun_dsa.preparar_escala_color_dsa(
    dsa_fa_plot,
    vmin=49,
    vmax=94,
    gamma=1
)

## 1. 5. Comprobaciones

In [ ]:
print("DSA f_a:", dsa_fa_plot.shape)
print("DSA EEG:", dsa_eeg_directa_plot.shape)

print("¿Tiempos iguales?")
print((tiempo_fa_unilat.reset_index(drop=True) == tiempo_eeg.reset_index(drop=True)).all())

print("Rango DSA EEG reconstruida:")
print(np.nanmin(dsa_eeg_directa_plot.values), np.nanmax(dsa_eeg_directa_plot.values))

print("Rango DSA f_a:")
print(np.nanmin(dsa_fa_plot.values), np.nanmax(dsa_fa_plot.values))

### 1. 5. 1.  Comparación base

In [ ]:
dsa_eeg_comparacion, dsa_fa_comparacion = fun_dsa.preparar_matrices_para_comparacion(
    dsa_eeg_directa_plot,
    dsa_fa_plot
)

dsa_eeg_z = fun_dsa.zscore_global(dsa_eeg_comparacion)
dsa_fa_z = fun_dsa.zscore_global(dsa_fa_comparacion)

metricas_base = fun_dsa.comparar_dsa_global(
    dsa_eeg_z,
    dsa_fa_z
)

metricas_base

## 1. 6. Suavizado + shift

In [ ]:
ventanas_sp_smooth = [5, 10, 30, 60]

df_suav_shift = fun_dsa.probar_suavizado_y_shifts(
    dsa_eeg_comparacion,
    dsa_fa_comparacion,
    ventanas_suavizado=ventanas_sp_smooth,
    shifts=range(0, 31)
)

df_suav_shift.sort_values("Pearson", ascending=False).head(7)

### 1. 6. 1. Tabla resumen final

In [ ]:
mejor_pearson = df_suav_shift.sort_values("Pearson", ascending=False).iloc[0]
mejor_spearman = df_suav_shift.sort_values("Spearman", ascending=False).iloc[0]

df_resumen_final = pd.DataFrame([
    {
        "criterio": "Mejor Pearson",
        "suavizado_s": mejor_pearson["suavizado_s"],
        "shift_s": mejor_pearson["shift_s"],
        "Pearson": mejor_pearson["Pearson"],
        "Spearman": mejor_pearson["Spearman"],
        "MAE": mejor_pearson["MAE"],
        "RMSE": mejor_pearson["RMSE"],
    },
    {
        "criterio": "Mejor Spearman",
        "suavizado_s": mejor_spearman["suavizado_s"],
        "shift_s": mejor_spearman["shift_s"],
        "Pearson": mejor_spearman["Pearson"],
        "Spearman": mejor_spearman["Spearman"],
        "MAE": mejor_spearman["MAE"],
        "RMSE": mejor_spearman["RMSE"],
    }
])

df_resumen_final

# 2. Bilateral - Advanced

In [ ]:
#archivo 17 horas
ruta_fa_bilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/bilateral/M-Py5D-04301923/DH04301923/L04301923.f_a"
ruta_spa_bilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/bilateral/M-Py5D-04301923/DH04301923/L04301923.spa"
ruta_ha_bilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/bilateral/M-Py5D-04301923/DH04301923/L04301923.h_a"
ruta_ta_bilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/bilateral/M-Py5D-04301923/DH04301923/L04301923.t_a"

archivo_r4a = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/bilateral/M-Py5D-04301923/DH04301923/L04301923.r4a" 

In [ ]:
"""
ruta_fa_bilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/bilateral/L05141322/L05141322.f_a"
ruta_spa_bilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/bilateral/L05141322/L05141322.spa"
ruta_ha_bilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/bilateral/L05141322/L05141322.h_a"
ruta_ta_bilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/bilateral/L05141322/L05141322.t_a"

archivo_r4a = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_advanced/bilateral/L05141322/L05141322.r4a" 
"""

In [ ]:
"""
# este es para el bis vista que no tiene fa
ruta_spa_bilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_antiguo/L04141330/L04141330.spa"
ruta_ha_bilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_antiguo/L04141330/L04141330.h_a"
ruta_ta_bilat = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_antiguo/L04141330/L04141330.t_a"

archivo_r4a = "C:/Users/usuario/TFG_BIS_GIS/data/data_bis_antiguo/L04141330/L04141330.r4a"
"""

## 2. 1. Archivo Espectral .f_a

In [ ]:
tiempo_fa_bilat, dsa_fa_L, dsa_fa_R = fau.cargar_fa_bilateral(
    ruta_fa_bilat,
    escalar_db=True
)

print("Dimensiones del archivo hemisferio izquierdo:", dsa_fa_L.shape)
print("Dimensiones del archivo hemisferio derecho:", dsa_fa_R.shape)

## 2. 2. Archivo variables procesadas .spa

In [ ]:
df_spa_raw_bilat = fau.procesar_spa(ruta_spa_bilat)
df_spa_bilat = fun_dsa_b.limpiar_spa_bilateral(df_spa_raw_bilat)

print("Dimensiones del archivo procesado:", df_spa_bilat.shape)
print(df_spa_bilat.columns.tolist())
display(df_spa_bilat.head())

### timeline oficial del .spa

In [ ]:
timeline_spa_bilat = fun_dsa_b.preparar_timeline_spa(
    df_spa=df_spa_bilat,
    columna_time="Time",
    resolver_duplicados="last",
    verbose=True
)

### ajustar .f_a a la timeline del .spa

In [ ]:
dsa_fa_L_spa = fun_dsa_b.ajustar_dsa_a_timeline_spa(
    tiempo_dsa=tiempo_fa_bilat,
    dsa=dsa_fa_L,
    timeline_spa=timeline_spa_bilat,
    nombre="f_a izquierdo",
    verbose=True
)

dsa_fa_R_spa = fun_dsa_b.ajustar_dsa_a_timeline_spa(
    tiempo_dsa=tiempo_fa_bilat,
    dsa=dsa_fa_R,
    timeline_spa=timeline_spa_bilat,
    nombre="f_a derecho",
    verbose=True
)


### Extraer .spa para cada hemisferio 

In [ ]:
df_spa_L = fun_dsa_b.extraer_lado_spa_bilateral(
    df_spa_bilat,
    lado="izq",
    verbose=False
)

df_spa_R = fun_dsa_b.extraer_lado_spa_bilateral(
    df_spa_bilat,
    lado="der",
    verbose=False
)

###  Fusión .f_a y .spa

In [ ]:
"""
Si hay dos paquetes con el mismo segundo, conservar el último suele ser razonable porque representa el estado más reciente dentro de ese segundo.

No usaría mean como opción principal porque hay columnas que no son realmente promediables
"""

df_merge_fa_L = fun_dsa.alinear_spa_con_tiempo(
    timeline_spa_bilat,
    df_spa_L,
    resolver_duplicados="last"
)

df_merge_fa_R = fun_dsa.alinear_spa_con_tiempo(
    timeline_spa_bilat,
    df_spa_R,
    resolver_duplicados="last"
)

# Curvas del BIS por hemisferio
sef_fa_L = df_merge_fa_L["SEF08"]
mef_fa_L = df_merge_fa_L["MEDFRQ08"]

sef_fa_R = df_merge_fa_R["SEF08"]
mef_fa_R = df_merge_fa_R["MEDFRQ08"]

In [ ]:
print("timeline_spa_bilat:", len(timeline_spa_bilat))
print("df_merge_fa_L:", df_merge_fa_L.shape)
print("df_merge_fa_R:", df_merge_fa_R.shape)

### Cabecera

In [ ]:
num_canales_bil, fs_bil, pendiente_bil, offset_bil = fau.extraer_parametros_eeg(ruta_ha_bilat)
print("Parámetros extraídos con éxito:")
print(f" - Canales: {num_canales_bil}")
print(f" - Frecuencia (Hz): {fs_bil}")
print(f" - Pendiente (m): {pendiente_bil:.8f}")
print(f" - Offset (b): {offset_bil:.4f}")

## 2. 3. Archivo ondas crudas .r4a

In [ ]:
df_eeg_bilateral = fun_dsa_b.leer_r4a(
    archivo_r4a, 
    pendiente_bil,
    offset_bil, 
    fs=fs_bil)

In [ ]:
# Alinear raw al tamaño y timeline del .spa
df_eeg_bilat_recortado, timeline_spa_bilat, info_alineacion = (
    fun_dsa_b.recortar_raw_segun_ta_y_spa(
        df_raw=df_eeg_bilateral,
        ruta_ta=ruta_ta_bilat,
        df_spa=df_spa_bilat,
        columna_time="Time",
        fs=fs_bil,
        resolver_duplicados="last",
        verbose=True
    )
)

# Usar el inicio del .spa como inicio de la DSA reconstruida
hora_inicio = timeline_spa_bilat.iloc[0]

print("Raw bilateral original:", df_eeg_bilateral.shape)
print("Raw bilateral recortado:", df_eeg_bilat_recortado.shape)
print("Timeline SPA:", len(timeline_spa_bilat))

### 2. 3. 1. Reconstrucción

In [ ]:
# ============================================================
# Reconstrucción DSA por canal
# ============================================================

ventana_welch_s = 1
paso_welch_s = 1

df_dsa_canal1, frecuencias_c1 = fun_dsa.crear_matriz_dsa_fft_welch_desde_eeg(
    df_eeg_bilat_recortado,
    "canal_1_uV",
    fs=fs_bil,
    ventana_seg=ventana_welch_s,
    paso_seg=paso_welch_s,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5,
    modo="densidad",
    tiempo_referencia="inicio"
)

df_dsa_canal2, _ = fun_dsa.crear_matriz_dsa_fft_welch_desde_eeg(
    df_eeg_bilat_recortado,
    "canal_2_uV",
    fs=fs_bil,
    ventana_seg=ventana_welch_s,
    paso_seg=paso_welch_s,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5,
    modo="densidad",
    tiempo_referencia="inicio"
)

df_dsa_canal3, _ = fun_dsa.crear_matriz_dsa_fft_welch_desde_eeg(
    df_eeg_bilat_recortado,
    "canal_3_uV",
    fs=fs_bil,
    ventana_seg=ventana_welch_s,
    paso_seg=paso_welch_s,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5,
    modo="densidad",
    tiempo_referencia="inicio"
)

df_dsa_canal4, _ = fun_dsa.crear_matriz_dsa_fft_welch_desde_eeg(
    df_eeg_bilat_recortado,
    "canal_4_uV",
    fs=fs_bil,
    ventana_seg=ventana_welch_s,
    paso_seg=paso_welch_s,
    fmin=0.5,
    fmax=30.0,
    paso_freq=0.5,
    modo="densidad",
    tiempo_referencia="inicio"
)

In [ ]:
# Combinación por hemisferio en escala lineal

cols_freq = [c for c in df_dsa_canal1.columns if c != "tiempo_s"]

# Izquierda: canal 1 + canal 2
pot_media_izq = (
    df_dsa_canal1[cols_freq].to_numpy(dtype=float) +
    df_dsa_canal2[cols_freq].to_numpy(dtype=float)
) / 2

# Derecha: canal 3 + canal 4
pot_media_der = (
    df_dsa_canal3[cols_freq].to_numpy(dtype=float) +
    df_dsa_canal4[cols_freq].to_numpy(dtype=float)
) / 2

df_dsa_izq = df_dsa_canal1.copy()
df_dsa_der = df_dsa_canal3.copy()

In [ ]:
# Conversión a dB para visualización

ref_potencia = 0.0001

df_dsa_izq[cols_freq] = 10 * np.log10(
    (pot_media_izq + 1e-12) / (ref_potencia ** 2)
)

df_dsa_der[cols_freq] = 10 * np.log10(
    (pot_media_der + 1e-12) / (ref_potencia ** 2)
)

### 2. 3. 2. Adaptación temporal, máscara y plot de DSA EEG

In [ ]:
# ============================================================
# Adaptación temporal inicial de la DSA reconstruida
# ============================================================

hora_inicio = timeline_spa_bilat.iloc[0]

tiempo_eeg_bilat_tmp, dsa_eeg_izq_tmp = fun_dsa.adaptar_dsa_reconstruida_para_plot(
    df_dsa=df_dsa_izq,
    frecuencias=frecuencias_c1,
    hora_inicio=hora_inicio,
    insertar_fila_inicial_nan=False
)

_, dsa_eeg_der_tmp = fun_dsa.adaptar_dsa_reconstruida_para_plot(
    df_dsa=df_dsa_der,
    frecuencias=frecuencias_c1,
    hora_inicio=hora_inicio,
    insertar_fila_inicial_nan=False
)

In [ ]:
# ============================================================
# Ajustar reconstruida a la timeline oficial del .spa
# ============================================================

dsa_eeg_izq_spa = fun_dsa_b.ajustar_dsa_a_timeline_spa(
    tiempo_dsa=tiempo_eeg_bilat_tmp,
    dsa=dsa_eeg_izq_tmp,
    timeline_spa=timeline_spa_bilat,
    nombre="DSA EEG reconstruida izquierda",
    verbose=True
)

dsa_eeg_der_spa = fun_dsa_b.ajustar_dsa_a_timeline_spa(
    tiempo_dsa=tiempo_eeg_bilat_tmp,
    dsa=dsa_eeg_der_tmp,
    timeline_spa=timeline_spa_bilat,
    nombre="DSA EEG reconstruida derecha",
    verbose=True
)

# Desde aquí, el tiempo oficial de la reconstruida también es el del .spa
tiempo_eeg_bilat = timeline_spa_bilat.copy()

In [ ]:
# Merge del .spa para la reconstruida

df_merge_eeg_izq = fun_dsa.alinear_spa_con_tiempo(
    tiempo=timeline_spa_bilat,
    df_spa=df_spa_L,
    resolver_duplicados="last"
)

df_merge_eeg_der = fun_dsa.alinear_spa_con_tiempo(
    tiempo=timeline_spa_bilat,
    df_spa=df_spa_R,
    resolver_duplicados="last"
)

### Máscaras de calidad bilateral - común

In [ ]:
_, mask_comun_L = fun_dsa.preparar_dsa_con_mask(
    tiempo=timeline_spa_bilat,
    dsa=dsa_eeg_izq_spa,
    df_merge=df_merge_eeg_izq,
    umbral_sqi=15,
    umbral_ceros=0.9,
    incluir_filas_nan=True
)

_, mask_comun_R = fun_dsa.preparar_dsa_con_mask(
    tiempo=timeline_spa_bilat,
    dsa=dsa_eeg_der_spa,
    df_merge=df_merge_eeg_der,
    umbral_sqi=15,
    umbral_ceros=0.9,
    incluir_filas_nan=True
)

print(f"Filas totales izquierda:", len(mask_comun_L))
print(f"Filas totales derecha:", len(mask_comun_R))

print(f"\nFilas enmascaradas izquierda:", mask_comun_L.sum())
print("Filas enmascaradas derecha:", mask_comun_R.sum())

print(f"\nFilas válidas izquierda:", (~mask_comun_L).sum())
print("Filas válidas derecha:", (~mask_comun_R).sum())

#### Añadir posibles huecos del .f_a, si existe archivo espectral

In [ ]:
# Filas completamente NaN del .f_a ajustado al .spa (comprobación adicional por si acaso)
mask_fa_nan_L = dsa_fa_L_spa.isna().all(axis=1)
mask_fa_nan_R = dsa_fa_R_spa.isna().all(axis=1)

mask_comun_L = mask_comun_L | mask_fa_nan_L
mask_comun_R = mask_comun_R | mask_fa_nan_R

In [ ]:
print("NaN f_a L:", mask_fa_nan_L.sum())
print("NaN f_a R:", mask_fa_nan_R.sum())

print("\nFilas enmascaradas finales L:", mask_comun_L.sum())
print("Filas enmascaradas finales R:", mask_comun_R.sum())

print("\nFilas válidas finales L:", (~mask_comun_L).sum())
print("Filas válidas finales R:", (~mask_comun_R).sum())

#### Aplicación

In [ ]:
# Aplicar máscara común para que izquierda y derecha tengan bandas blancas alineadas

# -------------------------------------- Para EEG ----------------------------------
dsa_eeg_izq_plot = dsa_eeg_izq_spa.copy()
dsa_eeg_der_plot = dsa_eeg_der_spa.copy()

dsa_eeg_izq_plot.loc[mask_comun_L.values, :] = np.nan
dsa_eeg_der_plot.loc[mask_comun_R.values, :] = np.nan

In [ ]:
# -------------------------------------- Para FA ----------------------------------
dsa_plot_fa_L_mask = dsa_fa_L_spa.copy()
dsa_plot_fa_R_mask = dsa_fa_R_spa.copy()

dsa_plot_fa_L_mask.loc[mask_comun_L.values, :] = np.nan
dsa_plot_fa_R_mask.loc[mask_comun_R.values, :] = np.nan

#### Comprobación

In [ ]:
# ============================================================
# Comprobación de bandas blancas tras aplicar máscaras por hemisferio
# ============================================================

# 1. Detectar filas completamente blancas en cada matriz
mask_blanca_eeg_L = dsa_eeg_izq_plot.isna().all(axis=1)
mask_blanca_eeg_R = dsa_eeg_der_plot.isna().all(axis=1)

mask_blanca_fa_L = dsa_plot_fa_L_mask.isna().all(axis=1)
mask_blanca_fa_R = dsa_plot_fa_R_mask.isna().all(axis=1)


# 2. Comprobar que las bandas blancas coinciden con la máscara aplicada
print("=== COMPROBACIÓN IZQUIERDA ===")
print("EEG L coincide con mask_comun_L:", (mask_blanca_eeg_L.values == mask_comun_L.values).all())
print("FA  L coincide con mask_comun_L:", (mask_blanca_fa_L.values == mask_comun_L.values).all())

print("Bandas blancas EEG L:", mask_blanca_eeg_L.sum())
print("Bandas blancas FA  L:", mask_blanca_fa_L.sum())
print("Filas en mask_comun_L:", mask_comun_L.sum())

print("Diferencias EEG L vs máscara:", (mask_blanca_eeg_L.values != mask_comun_L.values).sum())
print("Diferencias FA  L vs máscara:", (mask_blanca_fa_L.values != mask_comun_L.values).sum())


print("\n=== COMPROBACIÓN DERECHA ===")
print("EEG R coincide con mask_comun_R:", (mask_blanca_eeg_R.values == mask_comun_R.values).all())
print("FA  R coincide con mask_comun_R:", (mask_blanca_fa_R.values == mask_comun_R.values).all())

print("Bandas blancas EEG R:", mask_blanca_eeg_R.sum())
print("Bandas blancas FA  R:", mask_blanca_fa_R.sum())
print("Filas en mask_comun_R:", mask_comun_R.sum())

print("Diferencias EEG R vs máscara:", (mask_blanca_eeg_R.values != mask_comun_R.values).sum())
print("Diferencias FA  R vs máscara:", (mask_blanca_fa_R.values != mask_comun_R.values).sum())


# 3. Tabla para localizar posibles diferencias
df_check_blancas = pd.DataFrame({
    "Time": timeline_spa_bilat.reset_index(drop=True),

    "mask_comun_L": mask_comun_L.reset_index(drop=True),
    "blanca_eeg_L": mask_blanca_eeg_L.reset_index(drop=True),
    "blanca_fa_L": mask_blanca_fa_L.reset_index(drop=True),

    "mask_comun_R": mask_comun_R.reset_index(drop=True),
    "blanca_eeg_R": mask_blanca_eeg_R.reset_index(drop=True),
    "blanca_fa_R": mask_blanca_fa_R.reset_index(drop=True),
})

df_check_blancas["diff_eeg_L"] = df_check_blancas["mask_comun_L"] != df_check_blancas["blanca_eeg_L"]
df_check_blancas["diff_fa_L"] = df_check_blancas["mask_comun_L"] != df_check_blancas["blanca_fa_L"]

df_check_blancas["diff_eeg_R"] = df_check_blancas["mask_comun_R"] != df_check_blancas["blanca_eeg_R"]
df_check_blancas["diff_fa_R"] = df_check_blancas["mask_comun_R"] != df_check_blancas["blanca_fa_R"]


# 4. Mostrar solo filas problemáticas, si las hay
df_diferencias_blancas = df_check_blancas[
    df_check_blancas[
        ["diff_eeg_L", "diff_fa_L", "diff_eeg_R", "diff_fa_R"]
    ].any(axis=1)
]

print("\nNúmero total de filas con alguna diferencia:", len(df_diferencias_blancas))

display(df_diferencias_blancas.head(50))

### 2. 3. 3. Preparar escala de color de f_a y eeg reconstruida

In [ ]:
# -------------------------------------- Para EEG ----------------------------------
matriz_eeg_izq, matriz_eeg_der, vmin_eeg, vmax_eeg, norm_eeg_bilat, cmap_eeg_bilat = (
    fun_dsa_b.preparar_escala_color_dsa_bilateral(
        dsa_eeg_izq_plot,
        dsa_eeg_der_plot,
        gamma=0.55
    )
)

print("Reconstruida bilateral")
print("vmin:", vmin_eeg)
print("vmax:", vmax_eeg)

In [ ]:
# -------------------------------------- Para FA ----------------------------------
matriz_fa_L, matriz_fa_R, vmin_fa, vmax_fa, norm_fa_bilat, cmap_fa_bilat = (
    fun_dsa_b.preparar_escala_color_dsa_bilateral(
        dsa_plot_fa_L_mask,
        dsa_plot_fa_R_mask,
        vmin=49,
        vmax=94,
        gamma=1
    )
)


### 2. 3. 4. Visualizar reconstruidas desde .r4a

In [ ]:
fig_eeg, axes_eeg = fun_plot.plot_dsa_bilateral_con_sef_mef(
    tiempo=timeline_spa_bilat,
    frecuencias=dsa_eeg_izq_plot.columns.astype(float),
    matriz_izq=matriz_eeg_izq,
    matriz_der=matriz_eeg_der,
    norm=norm_eeg_bilat,
    cmap=cmap_eeg_bilat,
    df_merge_izq=df_merge_eeg_izq,
    df_merge_der=df_merge_eeg_der,
    mask_izq=mask_comun_L,
    mask_der=mask_comun_R,
    asimetria=df_merge_eeg_izq["ASYM09"] if "ASYM09" in df_merge_eeg_izq.columns else None,
    titulo_izq="DSA reconstruida desde EEG - Hemisferio izquierdo",
    titulo_der="DSA reconstruida desde EEG - Hemisferio derecho",
    titulo_general="DSA bilateral reconstruida desde archivo .r4a alineada a .spa",
    etiqueta_colorbar="Potencia espectral reconstruida (dB)"
)

### Máscaras y visualización - .f_a 

In [ ]:
frecuencias_fa = dsa_plot_fa_L_mask.columns.astype(float)

fig_fa, axes_fa = fun_plot.plot_dsa_bilateral_con_sef_mef(
    tiempo=timeline_spa_bilat,
    frecuencias=frecuencias_fa,
    matriz_izq=matriz_fa_L,
    matriz_der=matriz_fa_R,
    norm=norm_fa_bilat,
    cmap=cmap_fa_bilat,
    df_merge_izq=df_merge_fa_L,
    df_merge_der=df_merge_fa_R,
    mask_izq=mask_comun_L,
    mask_der=mask_comun_R,
    asimetria=df_merge_fa_L["ASYM09"],
    titulo_izq="DSA .f_a - Hemisferio izquierdo",
    titulo_der="DSA .f_a - Hemisferio derecho",
    titulo_general="DSA bilateral exportada en .f_a alineada a .spa",
    etiqueta_colorbar="Potencia espectral (dB)"
)

In [ ]:
print("timeline_spa_bilat:", len(timeline_spa_bilat))

print("FA L ajustado:", dsa_fa_L_spa.shape)
print("FA R ajustado:", dsa_fa_R_spa.shape)

print("EEG L ajustado:", dsa_eeg_izq_spa.shape)
print("EEG R ajustado:", dsa_eeg_der_spa.shape)

print("merge EEG L:", df_merge_eeg_izq.shape)
print("merge EEG R:", df_merge_eeg_der.shape)

print("plot EEG L:", dsa_eeg_izq_plot.shape)
print("plot EEG R:", dsa_eeg_der_plot.shape)

In [ ]:
cols_freq_eeg_L = [c for c in dsa_eeg_izq_plot.columns]

print("Mínimos y máximos hemisferio izquierdo - RECONSTRUCCIÓN")
print(np.nanmin(dsa_eeg_izq_plot[cols_freq_eeg_L].values))
print(np.nanmax(dsa_eeg_izq_plot[cols_freq_eeg_L].values))

print("Percentiles mínimos y máximos hemisferio izquierdo")
print(np.nanpercentile(dsa_eeg_izq_plot[cols_freq_eeg_L].values, 2))
print(np.nanpercentile(dsa_eeg_izq_plot[cols_freq_eeg_L].values, 99.5))

In [ ]:
cols_freq_fa_L = [c for c in dsa_plot_fa_L_mask.columns]

print("Mínimos y máximos hemisferio izquierdo - FA")
print(np.nanmin(dsa_plot_fa_L_mask[cols_freq_fa_L].values))
print(np.nanmax(dsa_plot_fa_L_mask[cols_freq_fa_L].values))

print("Percentiles mínimos y máximos hemisferio izquierdo")
print(np.nanpercentile(dsa_plot_fa_L_mask[cols_freq_fa_L].values, 2))
print(np.nanpercentile(dsa_plot_fa_L_mask[cols_freq_fa_L].values, 99.5))

In [ ]:
cols_freq_eeg_R = [c for c in dsa_eeg_der_plot.columns]

print("Mínimos y máximos hemisferio derecho - RECONSTRUCCIÓN")
print(np.nanmin(dsa_eeg_der_plot[cols_freq_eeg_R].values))
print(np.nanmax(dsa_eeg_der_plot[cols_freq_eeg_R].values))

print("Percentiles mínimos y máximos hemisferio derecho")
print(np.nanpercentile(dsa_eeg_der_plot[cols_freq_eeg_R].values, 2))
print(np.nanpercentile(dsa_eeg_der_plot[cols_freq_eeg_R].values, 99.5))

In [ ]:
cols_freq_fa_R = [c for c in dsa_plot_fa_R_mask.columns]

print("Mínimos y máximos hemisferio derecho - FA")
print(np.nanmin(dsa_plot_fa_R_mask[cols_freq_fa_R].values))
print(np.nanmax(dsa_plot_fa_R_mask[cols_freq_fa_R].values))

print("Percentiles mínimos y máximos hemisferio derecho")
print(np.nanpercentile(dsa_plot_fa_R_mask[cols_freq_fa_R].values, 2))
print(np.nanpercentile(dsa_plot_fa_R_mask[cols_freq_fa_R].values, 99.5))

# Métricas y comparación

``` python

# Para métricas base:
dsa_eeg_izq_plot      vs dsa_plot_fa_L_mask
dsa_eeg_der_plot      vs dsa_plot_fa_R_mask

# Para suavizado:
dsa_eeg_izq_spa  vs  dsa_fa_L_spa
dsa_eeg_der_spa  vs  dsa_fa_R_spa

# Para visualizar:
tiempo = timeline_spa_bilat
```

In [ ]:
# ============================================================
# Comparación base bilateral: EEG reconstruida vs .f_a
# ============================================================

# Izquierda
dsa_eeg_L_comp, dsa_fa_L_comp = fun_dsa.preparar_matrices_para_comparacion(
    dsa_eeg_izq_plot,
    dsa_plot_fa_L_mask
)

dsa_eeg_L_z = fun_dsa.zscore_global(dsa_eeg_L_comp)
dsa_fa_L_z = fun_dsa.zscore_global(dsa_fa_L_comp)

metricas_base_L = fun_dsa.comparar_dsa_global(
    dsa_eeg_L_z,
    dsa_fa_L_z
)


# Derecha
dsa_eeg_R_comp, dsa_fa_R_comp = fun_dsa.preparar_matrices_para_comparacion(
    dsa_eeg_der_plot,
    dsa_plot_fa_R_mask
)

dsa_eeg_R_z = fun_dsa.zscore_global(dsa_eeg_R_comp)
dsa_fa_R_z = fun_dsa.zscore_global(dsa_fa_R_comp)

metricas_base_R = fun_dsa.comparar_dsa_global(
    dsa_eeg_R_z,
    dsa_fa_R_z
)


# Resumen
# compara las matrices ya enmascaradas
df_metricas_base_bilat = pd.DataFrame([
    {"hemisferio": "izquierdo",
        **metricas_base_L},
    {"hemisferio": "derecho",
        **metricas_base_R}
])

display(df_metricas_base_bilat)

In [ ]:
# ============================================================
# Correlación por frecuencia bilateral
# ============================================================

df_corr_freq_L = fun_dsa.correlacion_por_frecuencia(
    dsa_eeg_L_z,
    dsa_fa_L_z
)

df_corr_freq_L["hemisferio"] = "izquierdo"


df_corr_freq_R = fun_dsa.correlacion_por_frecuencia(
    dsa_eeg_R_z,
    dsa_fa_R_z
)

df_corr_freq_R["hemisferio"] = "derecho"


df_corr_freq_bilat = pd.concat(
    [df_corr_freq_L, df_corr_freq_R],
    ignore_index=True
)

display(df_corr_freq_bilat.head())

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(
    df_corr_freq_L["frecuencia_Hz"],
    df_corr_freq_L["correlacion"],
    marker="o",
    label="Izquierdo"
)

plt.plot(
    df_corr_freq_R["frecuencia_Hz"],
    df_corr_freq_R["correlacion"],
    marker="o",
    label="Derecho"
)

for f in [4, 8, 13]:
    plt.axvline(f, color="gray", linestyle="--", linewidth=1, alpha=0.6)

plt.axhline(0, color="gray", linestyle="--", linewidth=1)

plt.xlabel("Frecuencia (Hz)")
plt.ylabel("Correlación")
plt.title("Correlación por frecuencia entre DSA EEG y DSA .f_a bilateral")
plt.legend()
plt.tight_layout()
plt.show()

### Suavizado + shift

In [ ]:
# ============================================================
# Suavizado + shift bilateral
# Usar matrices alineadas a .spa, pero SIN máscara visual aplicada
# ============================================================

ventanas_sp_smooth = [5, 10, 30, 60]
shifts_prueba = range(-10, 30)

# Izquierda
dsa_eeg_L_shift_comp, dsa_fa_L_shift_comp = fun_dsa.preparar_matrices_para_comparacion(
    dsa_eeg_izq_spa,
    dsa_fa_L_spa
)

df_suav_shift_L = fun_dsa.probar_suavizado_y_shifts(
    dsa_eeg_L_shift_comp,
    dsa_fa_L_shift_comp,
    ventanas_suavizado=ventanas_sp_smooth,
    shifts=shifts_prueba
)

df_suav_shift_L = df_suav_shift_L.rename(columns={
    "Pearson": "Pearson_L",
    "Spearman": "Spearman_L",
    "MAE": "MAE_L",
    "RMSE": "RMSE_L"
})


# Derecha
dsa_eeg_R_shift_comp, dsa_fa_R_shift_comp = fun_dsa.preparar_matrices_para_comparacion(
    dsa_eeg_der_spa,
    dsa_fa_R_spa
)

df_suav_shift_R = fun_dsa.probar_suavizado_y_shifts(
    dsa_eeg_R_shift_comp,
    dsa_fa_R_shift_comp,
    ventanas_suavizado=ventanas_sp_smooth,
    shifts=shifts_prueba
)

df_suav_shift_R = df_suav_shift_R.rename(columns={
    "Pearson": "Pearson_R",
    "Spearman": "Spearman_R",
    "MAE": "MAE_R",
    "RMSE": "RMSE_R"
})


# Unir resultados
df_suav_shift_bilat = df_suav_shift_L.merge(
    df_suav_shift_R,
    on=["suavizado_s", "shift_s"],
    how="inner"
)

df_suav_shift_bilat["Pearson_medio"] = (
    df_suav_shift_bilat["Pearson_L"] +
    df_suav_shift_bilat["Pearson_R"]
) / 2

df_suav_shift_bilat["Spearman_medio"] = (
    df_suav_shift_bilat["Spearman_L"] +
    df_suav_shift_bilat["Spearman_R"]
) / 2

df_suav_shift_bilat["MAE_medio"] = (
    df_suav_shift_bilat["MAE_L"] +
    df_suav_shift_bilat["MAE_R"]
) / 2

df_suav_shift_bilat["RMSE_medio"] = (
    df_suav_shift_bilat["RMSE_L"] +
    df_suav_shift_bilat["RMSE_R"]
) / 2

display(
    df_suav_shift_bilat
    .sort_values("Pearson_medio", ascending=False)
    .head(10)
)

### Elegir mejor suavizado + shift bilateral

In [ ]:
mejor_bilat_pearson = (
    df_suav_shift_bilat
    .sort_values("Pearson_medio", ascending=False)
    .iloc[0]
)

mejor_bilat_spearman = (
    df_suav_shift_bilat
    .sort_values("Spearman_medio", ascending=False)
    .iloc[0]
)

df_resumen_final_bilat = pd.DataFrame([
    {
        "criterio": "Mejor Pearson medio",
        "suavizado_s": mejor_bilat_pearson["suavizado_s"],
        "shift_s": mejor_bilat_pearson["shift_s"],
        "Pearson_L": mejor_bilat_pearson["Pearson_L"],
        "Pearson_R": mejor_bilat_pearson["Pearson_R"],
        "Pearson_medio": mejor_bilat_pearson["Pearson_medio"],
        "Spearman_L": mejor_bilat_pearson["Spearman_L"],
        "Spearman_R": mejor_bilat_pearson["Spearman_R"],
        "Spearman_medio": mejor_bilat_pearson["Spearman_medio"],
        "MAE_medio": mejor_bilat_pearson["MAE_medio"],
        "RMSE_medio": mejor_bilat_pearson["RMSE_medio"],
    },
    {
        "criterio": "Mejor Spearman medio",
        "suavizado_s": mejor_bilat_spearman["suavizado_s"],
        "shift_s": mejor_bilat_spearman["shift_s"],
        "Pearson_L": mejor_bilat_spearman["Pearson_L"],
        "Pearson_R": mejor_bilat_spearman["Pearson_R"],
        "Pearson_medio": mejor_bilat_spearman["Pearson_medio"],
        "Spearman_L": mejor_bilat_spearman["Spearman_L"],
        "Spearman_R": mejor_bilat_spearman["Spearman_R"],
        "Spearman_medio": mejor_bilat_spearman["Spearman_medio"],
        "MAE_medio": mejor_bilat_spearman["MAE_medio"],
        "RMSE_medio": mejor_bilat_spearman["RMSE_medio"],
    }
])

display(df_resumen_final_bilat)

In [ ]:
suavizado_final_bilat = int(mejor_bilat_pearson["suavizado_s"])
shift_final_bilat = int(mejor_bilat_pearson["shift_s"])

print("Suavizado final bilateral:", suavizado_final_bilat)
print("Shift final bilateral:", shift_final_bilat)

## Suavizado limpio

In [ ]:
# ============================================================
# Suavizado temporal bilateral
# ============================================================

suavizado_final_bilat = int(mejor_bilat_pearson["suavizado_s"])

dsa_eeg_izq_suav = dsa_eeg_izq_spa.copy().rolling(
    window=suavizado_final_bilat,
    min_periods=1,
    center=False
).mean()

dsa_eeg_der_suav = dsa_eeg_der_spa.copy().rolling(
    window=suavizado_final_bilat,
    min_periods=1,
    center=False
).mean()


### Aplicar máscara común al final

In [ ]:
dsa_eeg_izq_suav_plot = dsa_eeg_izq_suav.copy()
dsa_eeg_der_suav_plot = dsa_eeg_der_suav.copy()

dsa_eeg_izq_suav_plot.loc[mask_comun_L.values, :] = np.nan
dsa_eeg_der_suav_plot.loc[mask_comun_R.values, :] = np.nan

### Comprobación de bandas blancas

In [ ]:
# ============================================================
# Comprobación de bandas blancas
# ============================================================

mask_blanca_izq_directa = dsa_eeg_izq_plot.isna().all(axis=1)
mask_blanca_der_directa = dsa_eeg_der_plot.isna().all(axis=1)

mask_blanca_izq_suav = dsa_eeg_izq_suav_plot.isna().all(axis=1)
mask_blanca_der_suav = dsa_eeg_der_suav_plot.isna().all(axis=1)

print("Bandas blancas izquierda directa vs suavizada:")
print((mask_blanca_izq_directa == mask_blanca_izq_suav).all())
print("Diferencias izquierda:", (mask_blanca_izq_directa != mask_blanca_izq_suav).sum())

print("\nBandas blancas derecha directa vs suavizada:")
print((mask_blanca_der_directa == mask_blanca_der_suav).all())
print("Diferencias derecha:", (mask_blanca_der_directa != mask_blanca_der_suav).sum())

## Preparar escala de color común para suavizadas

In [ ]:
# ============================================================
# Escala común para DSA suavizadas izquierda/derecha
# ============================================================

matriz_suav_izq, matriz_suav_der, vmin_suav, vmax_suav, norm_suav_bilat, cmap_suav_bilat = (
    fun_dsa_b.preparar_escala_color_dsa_bilateral(
        dsa_eeg_izq_suav_plot,
        dsa_eeg_der_suav_plot,
        gamma=0.55
    )
)

print("Rango suavizada bilateral:")
print("vmin:", vmin_suav)
print("vmax:", vmax_suav)

In [ ]:
# ============================================================
# Comprobación de bandas blancas
# ============================================================

mask_blanca_izq_directa = dsa_eeg_izq_plot.isna().all(axis=1)
mask_blanca_der_directa = dsa_eeg_der_plot.isna().all(axis=1)

mask_blanca_izq_suav = dsa_eeg_izq_suav_plot.isna().all(axis=1)
mask_blanca_der_suav = dsa_eeg_der_suav_plot.isna().all(axis=1)

print("Bandas blancas izquierda directa vs suavizada:")
print((mask_blanca_izq_directa == mask_blanca_izq_suav).all())
print("Diferencias izquierda:", (mask_blanca_izq_directa != mask_blanca_izq_suav).sum())

print("\nBandas blancas derecha directa vs suavizada:")
print((mask_blanca_der_directa == mask_blanca_der_suav).all())
print("Diferencias derecha:", (mask_blanca_der_directa != mask_blanca_der_suav).sum())

## Aplicar shift bilateral manteniendo duración original

In [ ]:
# ============================================================
# Suavizado + shift bilateral manteniendo duración original
# ============================================================

shift_final_bilat = int(mejor_bilat_pearson["shift_s"])

dsa_eeg_izq_suav_shift_full = pd.DataFrame(
    np.nan,
    index=dsa_eeg_izq_suav.index,
    columns=dsa_eeg_izq_suav.columns
)

dsa_eeg_der_suav_shift_full = pd.DataFrame(
    np.nan,
    index=dsa_eeg_der_suav.index,
    columns=dsa_eeg_der_suav.columns
)

dsa_eeg_izq_suav_shift_full.iloc[shift_final_bilat:, :] = (
    dsa_eeg_izq_suav.iloc[:-shift_final_bilat, :].to_numpy()
)

dsa_eeg_der_suav_shift_full.iloc[shift_final_bilat:, :] = (
    dsa_eeg_der_suav.iloc[:-shift_final_bilat, :].to_numpy()
)

### Aplicar máscara común al final

In [ ]:
dsa_eeg_izq_suav_shift_full.loc[mask_comun_L.values, :] = np.nan
dsa_eeg_der_suav_shift_full.loc[mask_comun_R.values, :] = np.nan

In [ ]:
mask_blanca_izq_shift = dsa_eeg_izq_suav_shift_full.isna().all(axis=1)
mask_blanca_der_shift = dsa_eeg_der_suav_shift_full.isna().all(axis=1)

print("\nBandas blancas izquierda suavizada vs suavizada + shift:")
print((mask_blanca_izq_suav == mask_blanca_izq_shift).all())
print("Diferencias izquierda:", (mask_blanca_izq_suav != mask_blanca_izq_shift).sum())

print("\nBandas blancas derecha suavizada vs suavizada + shift:")
print((mask_blanca_der_suav == mask_blanca_der_shift).all())
print("Diferencias derecha:", (mask_blanca_der_suav != mask_blanca_der_shift).sum())

print("\nBandas blancas izquierda fa vs suavizada + shift:")
print((mask_blanca_izq_suav == mask_blanca_izq_shift).all())
print("Diferencias izquierda:", (mask_blanca_izq_suav != mask_blanca_izq_shift).sum())

print("\nBandas blancas derecha fa vs suavizada + shift:")
print((mask_blanca_der_suav == mask_blanca_der_shift).all())
print("Diferencias derecha:", (mask_blanca_der_suav != mask_blanca_der_shift).sum())

## Escala común para suavizada + shift

In [ ]:
# ============================================================
# Escala común para DSA suavizadas + shift
# ============================================================

matriz_shift_izq, matriz_shift_der, vmin_shift, vmax_shift, norm_shift_bilat, cmap_shift_bilat = (
    fun_dsa_b.preparar_escala_color_dsa_bilateral(
        dsa_eeg_izq_suav_shift_full,
        dsa_eeg_der_suav_shift_full,
        gamma=0.55
        
    )
)

print("Rango suavizada + shift bilateral:")
print("vmin:", vmin_shift)
print("vmax:", vmax_shift)

### Comprobación de formas

## Visualizar suavizada + shift bilateral

In [ ]:
fig, axes = fun_plot.plot_dsa_bilateral_con_sef_mef(
    tiempo=timeline_spa_bilat,
    frecuencias=dsa_eeg_izq_suav_shift_full.columns.astype(float),
    matriz_izq=matriz_shift_izq,
    matriz_der=matriz_shift_der,
    norm=norm_shift_bilat,
    cmap=cmap_shift_bilat,
    df_merge_izq=df_merge_eeg_izq,
    df_merge_der=df_merge_eeg_der,
    mask_izq=mask_comun_L,
    mask_der=mask_comun_R,
    asimetria=df_merge_eeg_izq["ASYM09"] if "ASYM09" in df_merge_eeg_izq.columns else None,
    titulo_izq="DSA reconstruida suavizada + shift - Hemisferio izquierdo",
    titulo_der="DSA reconstruida suavizada + shift - Hemisferio derecho",
    titulo_general="DSA bilateral reconstruida desde .r4a con suavizado y shift",
    etiqueta_colorbar="Intensidad espectral reconstruida (dB)"
)

In [ ]:
fig_fa, axes_fa = fun_plot.plot_dsa_bilateral_con_sef_mef(
    tiempo=timeline_spa_bilat,
    frecuencias=frecuencias_fa,
    matriz_izq=matriz_fa_L,
    matriz_der=matriz_fa_R,
    norm=norm_fa_bilat,
    cmap=cmap_fa_bilat,
    df_merge_izq=df_merge_fa_L,
    df_merge_der=df_merge_fa_R,
    mask_izq=mask_comun_L,
    mask_der=mask_comun_R,
    asimetria=df_merge_fa_L["ASYM09"],
    titulo_izq="DSA .f_a - Hemisferio izquierdo",
    titulo_der="DSA .f_a - Hemisferio derecho",
    titulo_general="DSA bilateral exportada en .f_a alineada a .spa",
    etiqueta_colorbar="Potencia espectral (dB)"
)

In [ ]:
fig_eeg, axes_eeg = fun_plot.plot_dsa_bilateral_con_sef_mef(
    tiempo=timeline_spa_bilat,
    frecuencias=dsa_eeg_izq_plot.columns.astype(float),
    matriz_izq=matriz_eeg_izq,
    matriz_der=matriz_eeg_der,
    norm=norm_eeg_bilat,
    cmap=cmap_eeg_bilat,
    df_merge_izq=df_merge_eeg_izq,
    df_merge_der=df_merge_eeg_der,
    mask_izq=mask_comun_L,
    mask_der=mask_comun_R,
    asimetria=df_merge_eeg_izq["ASYM09"] if "ASYM09" in df_merge_eeg_izq.columns else None,
    titulo_izq="DSA reconstruida desde EEG - Hemisferio izquierdo",
    titulo_der="DSA reconstruida desde EEG - Hemisferio derecho",
    titulo_general="DSA bilateral reconstruida desde archivo .r4a",
    etiqueta_colorbar="Potencia espectral reconstruida (dB)"
)